## Apartado 4. Análisis de subjetividad de los comentarios.

En esta sección vamos a emplear un modelo ya finetuneado para obtener el sentimiento y/o emoción que trenasmite el comentario.

Usaremos el modelo de hugging face **cardiffnlp/twitter-roberta-base-sentiment**.


In [1]:
import zstandard as zstd
import json
import argparse
import io
import sys
from pathlib import Path
from datetime import datetime, UTC
import random
import os
import pandas as pd
import numpy as np

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
"""
Función de prepocesado para poner placeholders en menciones y links. Usamos esta función
que viene dentro de la página del modelo de hugging face en vez de la usada en apartados anteriores
ya que posiblemente haya sido entrenada con estos placeholders y nos de mejores resultados.
 """
def preprocess(text):
    new_text = []


    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)


subreddits = ["onepiece", "soccer", "gaming", "movies", "leagueoflegends", "drawing"]

# Obtenemos el modelo paraa obtener los sentimientos
task='sentiment'
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer_sentiment = AutoTokenizer.from_pretrained(MODEL)

model_sentiment = AutoModelForSequenceClassification.from_pretrained(MODEL)

# Tambien nos guardamos a la vez el modelo para obtener las emociones
task='emotion'
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer_emotion= AutoTokenizer.from_pretrained(MODEL)

model_emotion = AutoModelForSequenceClassification.from_pretrained(MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emotion
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from scipy.special import softmax

def obtener_sentimientos(subreddits):
    labels_sentiment = {0: "Negative", 1: "Neutral", 2: "Positive"} # Orden de etiquetas sacada de la página de hugging face
    labels_emotions = {0: "anger", 1: "joy", 2: "optimism", 3: "sadness"}
    for subreddit in subreddits:
        filename = subreddit + "_final.json"
        output_path = subreddit + "_sentiments.json"

        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Recorremos las submissions
        for submission in data["submissions"]:
            comentarios = []
            # Recorremos cada comentario de todas las submissions de todos los subreddits
            for comment in submission["comments"]:
                sentiments = []
                emotions = []
                # Obtenemos el cuerpo de cada comentario
                cuerpo = str(comment.get("body", ""))
                if not cuerpo:
                    continue
                texto = preprocess(cuerpo)
                encoded_input_s = tokenizer_sentiment(texto, return_tensors='pt', truncation=True, max_length=512)

                # Primero vamos a extraer el sentimiento del comentario
                output_sentiment = model_sentiment(**encoded_input_s)
                scores_sentiment = output_sentiment[0][0].detach().numpy()
                scores_sentiment = softmax(scores_sentiment)
                # Ordenamos los sentimientos
                ranking_sentiment = np.argsort(scores_sentiment)
                ranking_sentiment = ranking_sentiment[::-1]
                # Nos quedamos con el que mayor score tiene
                best_sentiment = labels_sentiment[ranking_sentiment[0]]
                sentiments.append(best_sentiment)
                probs = {}
                # Nos guardamos también las probabilidades de cada sentimiento
                for i in range(scores_sentiment.shape[0]):
                    l = labels_sentiment[ranking_sentiment[i]]
                    s = scores_sentiment[ranking_sentiment[i]]
                    probs[l] = round(float(s), 4)
                sentiments.append(probs)
                comment["sentiments"] = sentiments

                # Ahora extraemos las emociones
                encoded_input_e = tokenizer_emotion(texto, return_tensors='pt', truncation=True, max_length=512)
                output_emotion = model_emotion(**encoded_input_e)
                scores_emotion = output_emotion[0][0].detach().numpy()
                scores_emotion = softmax(scores_emotion)
                # Ordenamos las emociones
                ranking_emotion = np.argsort(scores_emotion)
                ranking_emotion = ranking_emotion[::-1]
                # Nos quedamos con al emoción con mayor probabilidad
                best_emotion = labels_emotions[ranking_emotion[0]]
                emotions.append(best_emotion)
                probs = {}
                # Nos quedamos con todas las probabilidades
                for i in range(scores_emotion.shape[0]):
                    l = labels_emotions[ranking_emotion[i]]
                    s = scores_emotion[ranking_emotion[i]]
                    probs[l] = round(float(s), 4)
                emotions.append(probs)
                comment["emotions"] = emotions

                # Añadimos el comentario con los dos nuevos campos
                comentarios.append(comment)
            submission["comments"] = comentarios

        # Guardamos la nueva informción en un nuevo json
        with open(output_path, "w", encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

obtener_sentimientos(subreddits)


Una observación sobre las emociones que se asignan a los comentarios es que cuando el comentario es neutro en cuanto a la expresión de las 4 emociones, el modelo tiene una predisposición a clasificarlo como anger. Es posible que esta tendencia se deba a que la etiqueta de anger es la primera etiqueta (etiqueta 0).